In [ ]:
!pip uninstall -y numpy
!pip install -U "numpy==1.26.4" "transformers" "datasets" "accelerate" "sentencepiece" "scikit-learn" "pandas"

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you hav

In [ ]:
import numpy as np
print(np.__version__)

1.26.4


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [ ]:
import os
import shutil
import random
import inspect
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset, Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

# =========================================================
# CONFIG
# =========================================================
MODEL_NAME = "google/flan-t5-small"
HF_DATASET_REPO = "iberbench/iberbench_all"
HF_DATASET_CONFIG = "tass-tass-sentiment_analysis-2020-spanish-mexico"

OUTPUT_DIR = os.path.abspath("flan_t5_small_tass2020_mexico")

MAX_INPUT_LEN = 128
MAX_TARGET_LEN = 8
SEED = 42

BATCH_SIZE_TRAIN = 8
BATCH_SIZE_EVAL = 16
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 5
GRAD_ACCUM_STEPS = 1
WARMUP_STEPS = 20

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

PREFIX = (
    "Clasifica la polaridad del siguiente tweet en español de México. "
    "Responde solo con una palabra exacta de esta lista: "
    "positive, neutral, negative. Tweet: "
)

set_seed(SEED)

# =========================================================
# SEMILLAS
# =========================================================
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

# =========================================================
# MAPEO DE LABELS
# dataset observado:
# '0', '1', '2'
# asumimos:
# 0 -> negative
# 1 -> neutral
# 2 -> positive
# =========================================================
LABEL_ID_TO_TEXT = {
    0: "negative",
    1: "neutral",
    2: "positive",
}

def normalize_label_for_training(x):
    if isinstance(x, (int, np.integer)):
        if int(x) in LABEL_ID_TO_TEXT:
            return LABEL_ID_TO_TEXT[int(x)]
        raise ValueError(f"Etiqueta entera no reconocida: {x}")

    if isinstance(x, (float, np.floating)):
        if float(x).is_integer():
            xi = int(x)
            if xi in LABEL_ID_TO_TEXT:
                return LABEL_ID_TO_TEXT[xi]
        raise ValueError(f"Etiqueta float no reconocida: {x}")

    z = str(x).strip().lower()

    mapping = {
        "negative": "negative",
        "negativo": "negative",
        "negativa": "negative",
        "neg": "negative",
        "n": "negative",
        "0": "negative",
        "-1": "negative",

        "neutral": "neutral",
        "neutro": "neutral",
        "neutra": "neutral",
        "neu": "neutral",
        "none": "neutral",
        "1": "neutral",

        "positive": "positive",
        "positivo": "positive",
        "positiva": "positive",
        "pos": "positive",
        "p": "positive",
        "2": "positive",
    }

    if z in mapping:
        return mapping[z]

    if "neg" in z:
        return "negative"
    if "neu" in z or "neutr" in z or z == "none":
        return "neutral"
    if "pos" in z or "positiv" in z:
        return "positive"

    raise ValueError(f"Etiqueta no reconocida: {x}")


def normalize_generated_label(text: str) -> str:
    z = str(text).strip().lower()
    z = z.replace(".", "").replace(",", "").replace(":", "").replace(";", "").strip()

    alias = {
        "positive": "positive",
        "positivo": "positive",
        "positiva": "positive",
        "pos": "positive",
        "p": "positive",

        "negative": "negative",
        "negativo": "negative",
        "negativa": "negative",
        "neg": "negative",
        "n": "negative",

        "neutral": "neutral",
        "neutro": "neutral",
        "neutra": "neutral",
        "neu": "neutral",
    }

    if z in alias:
        return alias[z]

    for k, v in alias.items():
        if k in z:
            return v

    return "neutral"

# =========================================================
# DETECCIÓN DE COLUMNAS
# =========================================================
def guess_text_column(df: pd.DataFrame) -> str:
    candidates = ["text", "tweet", "content", "sentence", "document", "texto"]
    for c in candidates:
        if c in df.columns:
            return c

    for c in df.columns:
        if df[c].dtype == "object" and c.lower() not in {
            "label", "labels", "sentiment", "polarity", "class", "target"
        }:
            return c

    raise ValueError(f"No pude inferir la columna de texto. Columnas: {list(df.columns)}")


def guess_label_column(df: pd.DataFrame) -> str:
    candidates = ["label", "labels", "sentiment", "polarity", "class", "target", "gold"]
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"No pude inferir la columna de etiqueta. Columnas: {list(df.columns)}")

# =========================================================
# DESCARGA Y PREPARACIÓN DEL DATASET
# =========================================================
def load_and_prepare_dataset():
    print(f"[INFO] Descargando dataset: {HF_DATASET_REPO} / {HF_DATASET_CONFIG}")
    ds = load_dataset(HF_DATASET_REPO, HF_DATASET_CONFIG)

    print("\n[INFO] Splits disponibles:")
    print(ds)

    split_names = list(ds.keys())

    if "train" in split_names:
        train_raw = ds["train"].to_pandas()
    else:
        train_raw = ds[split_names[0]].to_pandas()

    valid_raw = ds["validation"].to_pandas() if "validation" in split_names else None
    test_raw = ds["test"].to_pandas() if "test" in split_names else None

    text_col = guess_text_column(train_raw)
    label_col = guess_label_column(train_raw)

    print(f"\n[INFO] Columna texto detectada: {text_col}")
    print(f"[INFO] Columna label detectada: {label_col}")

    print("\n[INFO] Labels únicos crudos en train:")
    print(sorted(train_raw[label_col].dropna().unique().tolist()))

    train_df = train_raw[[text_col, label_col]].copy()
    train_df.columns = ["text", "label"]
    train_df["text"] = train_df["text"].astype(str)
    train_df["label"] = train_df["label"].apply(normalize_label_for_training)

    if valid_raw is not None:
        valid_df = valid_raw[[text_col, label_col]].copy()
        valid_df.columns = ["text", "label"]
        valid_df["text"] = valid_df["text"].astype(str)
        valid_df["label"] = valid_df["label"].apply(normalize_label_for_training)
    else:
        train_df, valid_df = train_test_split(
            train_df,
            test_size=0.1,
            random_state=SEED,
            stratify=train_df["label"],
        )
        train_df = train_df.reset_index(drop=True)
        valid_df = valid_df.reset_index(drop=True)

    if test_raw is not None:
        test_df = test_raw[[text_col, label_col]].copy()
        test_df.columns = ["text", "label"]
        test_df["text"] = test_df["text"].astype(str)
        test_df["label"] = test_df["label"].apply(normalize_label_for_training)
    else:
        test_df = None

    return train_df, valid_df, test_df


train_df, valid_df, test_df = load_and_prepare_dataset()

print("\n==============================")
print("TRAIN SHAPE:", train_df.shape)
print("VALID SHAPE:", valid_df.shape)
print("TEST SHAPE :", None if test_df is None else test_df.shape)
print("==============================")

print("\nDistribución TRAIN:")
print(train_df["label"].value_counts())

print("\nDistribución VALID:")
print(valid_df["label"].value_counts())

if test_df is not None:
    print("\nDistribución TEST:")
    print(test_df["label"].value_counts())

# =========================================================
# DATASETDICT
# =========================================================
hf_ds = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True), preserve_index=False),
    "validation": Dataset.from_pandas(valid_df.reset_index(drop=True), preserve_index=False),
})

if test_df is not None:
    hf_ds["test"] = Dataset.from_pandas(test_df.reset_index(drop=True), preserve_index=False)

# =========================================================
# TOKENIZER Y MODELO
# =========================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)

# NO tocar tie_word_embeddings

# =========================================================
# PREPROCESAMIENTO
# =========================================================
def preprocess_function(examples):
    inputs = [PREFIX + str(x) for x in examples["text"]]
    targets = [str(y) for y in examples["label"]]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LEN,
        truncation=True,
    )

    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LEN,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized_datasets = hf_ds.map(
    preprocess_function,
    batched=True,
    remove_columns=hf_ds["train"].column_names,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)

# =========================================================
# MÉTRICAS
# =========================================================
def compute_metrics_from_texts(gold, pred):
    return {
        "accuracy": accuracy_score(gold, pred),
        "f1_macro": f1_score(gold, pred, average="macro"),
        "f1_weighted": f1_score(gold, pred, average="weighted"),
    }

# =========================================================
# LIMPIAR SALIDA
# =========================================================
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

# =========================================================
# TRAINING ARGUMENTS COMPATIBLES Y SIMPLES
# =========================================================
def build_training_args():
    sig = inspect.signature(Seq2SeqTrainingArguments.__init__)
    valid_params = set(sig.parameters.keys())

    kwargs = {}

    def add_if_supported(name, value):
        if name in valid_params:
            kwargs[name] = value

    add_if_supported("output_dir", OUTPUT_DIR)
    add_if_supported("per_device_train_batch_size", BATCH_SIZE_TRAIN)
    add_if_supported("per_device_eval_batch_size", BATCH_SIZE_EVAL)
    add_if_supported("gradient_accumulation_steps", GRAD_ACCUM_STEPS)
    add_if_supported("learning_rate", LEARNING_RATE)
    add_if_supported("weight_decay", WEIGHT_DECAY)
    add_if_supported("num_train_epochs", NUM_EPOCHS)
    add_if_supported("warmup_steps", WARMUP_STEPS)
    add_if_supported("report_to", "none")
    add_if_supported("seed", SEED)
    add_if_supported("disable_tqdm", True)
    add_if_supported("save_total_limit", 1)
    add_if_supported("optim", "adamw_torch")
    add_if_supported("remove_unused_columns", True)

    # importante: NO fp16 para evitar inestabilidad
    add_if_supported("fp16", False)

    # no hacemos evaluación interna del trainer
    if "eval_strategy" in valid_params:
        kwargs["eval_strategy"] = "no"
    elif "evaluation_strategy" in valid_params:
        kwargs["evaluation_strategy"] = "no"

    if "save_strategy" in valid_params:
        kwargs["save_strategy"] = "no"

    if "logging_strategy" in valid_params:
        kwargs["logging_strategy"] = "steps"

    if "logging_steps" in valid_params:
        kwargs["logging_steps"] = 20

    add_if_supported("predict_with_generate", False)
    add_if_supported("load_best_model_at_end", False)

    return Seq2SeqTrainingArguments(**kwargs)


training_args = build_training_args()

print("\n[INFO] Training args:")
print(training_args)

# =========================================================
# TRAINER
# =========================================================
trainer_sig = inspect.signature(Seq2SeqTrainer.__init__)
trainer_valid_params = set(trainer_sig.parameters.keys())

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": tokenized_datasets["train"],
    "eval_dataset": tokenized_datasets["validation"],
    "data_collator": data_collator,
}

if "tokenizer" in trainer_valid_params:
    trainer_kwargs["tokenizer"] = tokenizer
elif "processing_class" in trainer_valid_params:
    trainer_kwargs["processing_class"] = tokenizer

trainer = Seq2SeqTrainer(**trainer_kwargs)

# =========================================================
# ENTRENAMIENTO
# =========================================================
print("\n[INFO] Iniciando entrenamiento...")
train_result = trainer.train()

print("\n=== TRAIN RESULT ===")
print(train_result)

# =========================================================
# GUARDAR MODELO FINAL
# =========================================================
os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"\n[OK] Modelo guardado en: {OUTPUT_DIR}")

# =========================================================
# INFERENCIA
# =========================================================
@torch.no_grad()
def predict_sentiment(text: str):
    model.eval()

    enc = tokenizer(
        PREFIX + str(text),
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LEN,
    )
    enc = {k: v.to(DEVICE) for k, v in enc.items()}

    out = model.generate(
        **enc,
        max_new_tokens=MAX_TARGET_LEN,
        do_sample=False,
    )

    raw = tokenizer.decode(out[0], skip_special_tokens=True).strip()
    pred = normalize_generated_label(raw)

    return {
        "text": text,
        "raw_output": raw,
        "label": pred,
    }


@torch.no_grad()
def predict_batch_texts(texts, batch_size=32):
    model.eval()
    all_preds = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        enc = tokenizer(
            [PREFIX + str(x) for x in batch_texts],
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=MAX_INPUT_LEN,
        )
        enc = {k: v.to(DEVICE) for k, v in enc.items()}

        out = model.generate(
            **enc,
            max_new_tokens=MAX_TARGET_LEN,
            do_sample=False,
        )

        decoded = tokenizer.batch_decode(out, skip_special_tokens=True)
        decoded = [normalize_generated_label(x) for x in decoded]
        all_preds.extend(decoded)

    return all_preds

# =========================================================
# EVALUACIÓN MANUAL
# =========================================================
print("\n=== EVALUACIÓN MANUAL / VALID ===")
valid_texts = valid_df["text"].astype(str).tolist()
valid_gold = valid_df["label"].astype(str).tolist()
valid_pred = predict_batch_texts(valid_texts, batch_size=BATCH_SIZE_EVAL)

metrics_valid = compute_metrics_from_texts(valid_gold, valid_pred)
print(metrics_valid)

print("\n=== CLASSIFICATION REPORT / VALID ===")
print(classification_report(valid_gold, valid_pred, digits=4))

if test_df is not None:
    print("\n=== EVALUACIÓN MANUAL / TEST ===")
    test_texts = test_df["text"].astype(str).tolist()
    test_gold = test_df["label"].astype(str).tolist()
    test_pred = predict_batch_texts(test_texts, batch_size=BATCH_SIZE_EVAL)

    metrics_test = compute_metrics_from_texts(test_gold, test_pred)
    print(metrics_test)

    print("\n=== CLASSIFICATION REPORT / TEST ===")
    print(classification_report(test_gold, test_pred, digits=4))

# =========================================================
# EJEMPLOS
# =========================================================
examples = [
    "Qué chido quedó eso",
    "Está de la fregada el servicio",
    "No está mal, cumple",
]

print("\n=== EJEMPLOS ===")
for ex in examples:
    print(predict_sentiment(ex))

[INFO] Descargando dataset: iberbench/iberbench_all / tass-tass-sentiment_analysis-2020-spanish-mexico

[INFO] Splits disponibles:
DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'language_variation', 'language'],
        num_rows: 989
    })
})

[INFO] Columna texto detectada: text
[INFO] Columna label detectada: label

[INFO] Labels únicos crudos en train:
['0', '1', '2']

TRAIN SHAPE: (890, 2)
VALID SHAPE: (99, 2)
TEST SHAPE : None

Distribución TRAIN:
label
negative    453
neutral     282
positive    155
Name: count, dtype: int64

Distribución VALID:
label
negative    51
neutral     31
positive    17
Name: count, dtype: int64


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Map:   0%|          | 0/890 [00:00<?, ? examples/s]

Map:   0%|          | 0/99 [00:00<?, ? examples/s]


[INFO] Training args:
Seq2SeqTrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=True,
do_eval=False,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.NO,


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


[OK] Modelo guardado en: /content/flan_t5_small_tass2020_mexico

=== EVALUACIÓN MANUAL / VALID ===
{'accuracy': 0.6868686868686869, 'f1_macro': 0.6082889856474761, 'f1_weighted': 0.67331271104856}

=== CLASSIFICATION REPORT / VALID ===
              precision    recall  f1-score   support

    negative     0.7455    0.8039    0.7736        51
     neutral     0.6286    0.7097    0.6667        31
    positive     0.5556    0.2941    0.3846        17

    accuracy                         0.6869        99
   macro avg     0.6432    0.6026    0.6083        99
weighted avg     0.6762    0.6869    0.6733        99


=== EJEMPLOS ===
{'text': 'Qué chido quedó eso', 'raw_output': 'positive', 'label': 'positive'}
{'text': 'Está de la fregada el servicio', 'raw_output': 'negative', 'label': 'negative'}
{'text': 'No está mal, cumple', 'raw_output': 'negative', 'label': 'negative'}


In [ ]:
import os
import shutil
import random
import inspect
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset, Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

# =========================================================
# CONFIG
# =========================================================
MODEL_NAME = "google/flan-t5-small"
HF_DATASET_REPO = "iberbench/iberbench_all"
HF_DATASET_CONFIG = "tass-tass-sentiment_analysis-2020-spanish-mexico"

OUTPUT_DIR = os.path.abspath("flan_t5_small_tass2020_mexico")

MAX_INPUT_LEN = 128
MAX_TARGET_LEN = 8
SEED = 42

BATCH_SIZE_TRAIN = 8
BATCH_SIZE_EVAL = 16
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 5
GRAD_ACCUM_STEPS = 1
WARMUP_STEPS = 20

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

PREFIX = (
    "Clasifica la polaridad del siguiente tweet en español de México. "
    "Responde solo con una palabra exacta de esta lista: "
    "positive, neutral, negative. Tweet: "
)

set_seed(SEED)

# =========================================================
# SEMILLAS
# =========================================================
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

# =========================================================
# MAPEO DE LABELS
# dataset observado:
# '0', '1', '2'
# asumimos:
# 0 -> negative
# 1 -> neutral
# 2 -> positive
# =========================================================
LABEL_ID_TO_TEXT = {
    0: "negative",
    1: "neutral",
    2: "positive",
}

def normalize_label_for_training(x):
    if isinstance(x, (int, np.integer)):
        if int(x) in LABEL_ID_TO_TEXT:
            return LABEL_ID_TO_TEXT[int(x)]
        raise ValueError(f"Etiqueta entera no reconocida: {x}")

    if isinstance(x, (float, np.floating)):
        if float(x).is_integer():
            xi = int(x)
            if xi in LABEL_ID_TO_TEXT:
                return LABEL_ID_TO_TEXT[xi]
        raise ValueError(f"Etiqueta float no reconocida: {x}")

    z = str(x).strip().lower()

    mapping = {
        "negative": "negative",
        "negativo": "negative",
        "negativa": "negative",
        "neg": "negative",
        "n": "negative",
        "0": "negative",
        "-1": "negative",

        "neutral": "neutral",
        "neutro": "neutral",
        "neutra": "neutral",
        "neu": "neutral",
        "none": "neutral",
        "1": "neutral",

        "positive": "positive",
        "positivo": "positive",
        "positiva": "positive",
        "pos": "positive",
        "p": "positive",
        "2": "positive",
    }

    if z in mapping:
        return mapping[z]

    if "neg" in z:
        return "negative"
    if "neu" in z or "neutr" in z or z == "none":
        return "neutral"
    if "pos" in z or "positiv" in z:
        return "positive"

    raise ValueError(f"Etiqueta no reconocida: {x}")


def normalize_generated_label(text: str) -> str:
    z = str(text).strip().lower()
    z = z.replace(".", "").replace(",", "").replace(":", "").replace(";", "").strip()

    alias = {
        "positive": "positive",
        "positivo": "positive",
        "positiva": "positive",
        "pos": "positive",
        "p": "positive",

        "negative": "negative",
        "negativo": "negative",
        "negativa": "negative",
        "neg": "negative",
        "n": "negative",

        "neutral": "neutral",
        "neutro": "neutral",
        "neutra": "neutral",
        "neu": "neutral",
    }

    if z in alias:
        return alias[z]

    for k, v in alias.items():
        if k in z:
            return v

    return "neutral"

# =========================================================
# DETECCIÓN DE COLUMNAS
# =========================================================
def guess_text_column(df: pd.DataFrame) -> str:
    candidates = ["text", "tweet", "content", "sentence", "document", "texto"]
    for c in candidates:
        if c in df.columns:
            return c

    for c in df.columns:
        if df[c].dtype == "object" and c.lower() not in {
            "label", "labels", "sentiment", "polarity", "class", "target"
        }:
            return c

    raise ValueError(f"No pude inferir la columna de texto. Columnas: {list(df.columns)}")


def guess_label_column(df: pd.DataFrame) -> str:
    candidates = ["label", "labels", "sentiment", "polarity", "class", "target", "gold"]
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"No pude inferir la columna de etiqueta. Columnas: {list(df.columns)}")

# =========================================================
# DESCARGA Y PREPARACIÓN DEL DATASET
# =========================================================
def load_and_prepare_dataset():
    print(f"[INFO] Descargando dataset: {HF_DATASET_REPO} / {HF_DATASET_CONFIG}")
    ds = load_dataset(HF_DATASET_REPO, HF_DATASET_CONFIG)

    print("\n[INFO] Splits disponibles:")
    print(ds)

    split_names = list(ds.keys())

    if "train" in split_names:
        train_raw = ds["train"].to_pandas()
    else:
        train_raw = ds[split_names[0]].to_pandas()

    valid_raw = ds["validation"].to_pandas() if "validation" in split_names else None
    test_raw = ds["test"].to_pandas() if "test" in split_names else None

    text_col = guess_text_column(train_raw)
    label_col = guess_label_column(train_raw)

    print(f"\n[INFO] Columna texto detectada: {text_col}")
    print(f"[INFO] Columna label detectada: {label_col}")

    print("\n[INFO] Labels únicos crudos en train:")
    print(sorted(train_raw[label_col].dropna().unique().tolist()))

    train_df = train_raw[[text_col, label_col]].copy()
    train_df.columns = ["text", "label"]
    train_df["text"] = train_df["text"].astype(str)
    train_df["label"] = train_df["label"].apply(normalize_label_for_training)

    if valid_raw is not None:
        valid_df = valid_raw[[text_col, label_col]].copy()
        valid_df.columns = ["text", "label"]
        valid_df["text"] = valid_df["text"].astype(str)
        valid_df["label"] = valid_df["label"].apply(normalize_label_for_training)
    else:
        train_df, valid_df = train_test_split(
            train_df,
            test_size=0.1,
            random_state=SEED,
            stratify=train_df["label"],
        )
        train_df = train_df.reset_index(drop=True)
        valid_df = valid_df.reset_index(drop=True)

    if test_raw is not None:
        test_df = test_raw[[text_col, label_col]].copy()
        test_df.columns = ["text", "label"]
        test_df["text"] = test_df["text"].astype(str)
        test_df["label"] = test_df["label"].apply(normalize_label_for_training)
    else:
        test_df = None

    return train_df, valid_df, test_df


train_df, valid_df, test_df = load_and_prepare_dataset()

print("\n==============================")
print("TRAIN SHAPE:", train_df.shape)
print("VALID SHAPE:", valid_df.shape)
print("TEST SHAPE :", None if test_df is None else test_df.shape)
print("==============================")

print("\nDistribución TRAIN:")
print(train_df["label"].value_counts())

print("\nDistribución VALID:")
print(valid_df["label"].value_counts())

if test_df is not None:
    print("\nDistribución TEST:")
    print(test_df["label"].value_counts())

# =========================================================
# DATASETDICT
# =========================================================
hf_ds = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True), preserve_index=False),
    "validation": Dataset.from_pandas(valid_df.reset_index(drop=True), preserve_index=False),
})

if test_df is not None:
    hf_ds["test"] = Dataset.from_pandas(test_df.reset_index(drop=True), preserve_index=False)

# =========================================================
# TOKENIZER Y MODELO BASE
# =========================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)

# =========================================================
# PREPROCESAMIENTO
# =========================================================
def preprocess_function(examples):
    inputs = [PREFIX + str(x) for x in examples["text"]]
    targets = [str(y) for y in examples["label"]]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LEN,
        truncation=True,
    )

    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LEN,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized_datasets = hf_ds.map(
    preprocess_function,
    batched=True,
    remove_columns=hf_ds["train"].column_names,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)

# =========================================================
# UTILIDADES DE EVALUACIÓN / INFERENCIA
# =========================================================
@torch.no_grad()
def predict_batch_texts(model, tokenizer, texts, batch_size=32):
    model.eval()
    all_preds = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        enc = tokenizer(
            [PREFIX + str(x) for x in batch_texts],
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=MAX_INPUT_LEN,
        )
        enc = {k: v.to(DEVICE) for k, v in enc.items()}

        out = model.generate(
            **enc,
            max_new_tokens=MAX_TARGET_LEN,
            do_sample=False,
        )

        decoded = tokenizer.batch_decode(out, skip_special_tokens=True)
        decoded = [normalize_generated_label(x) for x in decoded]
        all_preds.extend(decoded)

    return all_preds


@torch.no_grad()
def predict_sentiment(model, tokenizer, text: str):
    model.eval()

    enc = tokenizer(
        PREFIX + str(text),
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LEN,
    )
    enc = {k: v.to(DEVICE) for k, v in enc.items()}

    out = model.generate(
        **enc,
        max_new_tokens=MAX_TARGET_LEN,
        do_sample=False,
    )

    raw = tokenizer.decode(out[0], skip_special_tokens=True).strip()
    pred = normalize_generated_label(raw)

    return {
        "text": text,
        "raw_output": raw,
        "label": pred,
    }


def compute_metrics_from_texts(gold, pred):
    return {
        "accuracy": accuracy_score(gold, pred),
        "f1_macro": f1_score(gold, pred, average="macro"),
        "f1_weighted": f1_score(gold, pred, average="weighted"),
    }


def evaluate_model_on_df(model, tokenizer, df, split_name="VALID", batch_size=16):
    texts = df["text"].astype(str).tolist()
    gold = df["label"].astype(str).tolist()
    pred = predict_batch_texts(model, tokenizer, texts, batch_size=batch_size)

    metrics = compute_metrics_from_texts(gold, pred)

    print(f"\n=== EVALUACIÓN MANUAL / {split_name} ===")
    print(metrics)

    print(f"\n=== CLASSIFICATION REPORT / {split_name} ===")
    print(classification_report(gold, pred, digits=4))

    out_df = pd.DataFrame({
        "text": texts,
        "gold": gold,
        "pred": pred,
    })

    return metrics, out_df


# =========================================================
# BASELINE SIN FINE-TUNING
# =========================================================
print("\n" + "=" * 70)
print("BASELINE / PRE-FINETUNE")
print("=" * 70)

baseline_valid_metrics, baseline_valid_df = evaluate_model_on_df(
    model=model,
    tokenizer=tokenizer,
    df=valid_df,
    split_name="VALID / PRE-FINETUNE",
    batch_size=BATCH_SIZE_EVAL,
)

if test_df is not None:
    baseline_test_metrics, baseline_test_df = evaluate_model_on_df(
        model=model,
        tokenizer=tokenizer,
        df=test_df,
        split_name="TEST / PRE-FINETUNE",
        batch_size=BATCH_SIZE_EVAL,
    )
else:
    baseline_test_metrics, baseline_test_df = None, None

print("\n=== EJEMPLOS / PRE-FINETUNE ===")
examples = [
    "Qué chido quedó eso",
    "Está de la fregada el servicio",
    "No está mal, cumple",
]
for ex in examples:
    print(predict_sentiment(model, tokenizer, ex))

# =========================================================
# LIMPIAR SALIDA
# =========================================================
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

# =========================================================
# TRAINING ARGUMENTS COMPATIBLES
# =========================================================
def build_training_args():
    sig = inspect.signature(Seq2SeqTrainingArguments.__init__)
    valid_params = set(sig.parameters.keys())

    kwargs = {}

    def add_if_supported(name, value):
        if name in valid_params:
            kwargs[name] = value

    add_if_supported("output_dir", OUTPUT_DIR)
    add_if_supported("per_device_train_batch_size", BATCH_SIZE_TRAIN)
    add_if_supported("per_device_eval_batch_size", BATCH_SIZE_EVAL)
    add_if_supported("gradient_accumulation_steps", GRAD_ACCUM_STEPS)
    add_if_supported("learning_rate", LEARNING_RATE)
    add_if_supported("weight_decay", WEIGHT_DECAY)
    add_if_supported("num_train_epochs", NUM_EPOCHS)
    add_if_supported("warmup_steps", WARMUP_STEPS)
    add_if_supported("report_to", "none")
    add_if_supported("seed", SEED)
    add_if_supported("disable_tqdm", True)
    add_if_supported("save_total_limit", 1)
    add_if_supported("optim", "adamw_torch")
    add_if_supported("remove_unused_columns", True)

    # configuración conservadora y estable
    add_if_supported("fp16", False)
    add_if_supported("predict_with_generate", False)
    add_if_supported("load_best_model_at_end", False)

    if "eval_strategy" in valid_params:
        kwargs["eval_strategy"] = "no"
    elif "evaluation_strategy" in valid_params:
        kwargs["evaluation_strategy"] = "no"

    if "save_strategy" in valid_params:
        kwargs["save_strategy"] = "no"

    if "logging_strategy" in valid_params:
        kwargs["logging_strategy"] = "steps"

    if "logging_steps" in valid_params:
        kwargs["logging_steps"] = 20

    return Seq2SeqTrainingArguments(**kwargs)


training_args = build_training_args()

print("\n[INFO] Training args:")
print(training_args)

# =========================================================
# TRAINER COMPATIBLE
# =========================================================
trainer_sig = inspect.signature(Seq2SeqTrainer.__init__)
trainer_valid_params = set(trainer_sig.parameters.keys())

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": tokenized_datasets["train"],
    "eval_dataset": tokenized_datasets["validation"],
    "data_collator": data_collator,
}

if "tokenizer" in trainer_valid_params:
    trainer_kwargs["tokenizer"] = tokenizer
elif "processing_class" in trainer_valid_params:
    trainer_kwargs["processing_class"] = tokenizer

trainer = Seq2SeqTrainer(**trainer_kwargs)

# =========================================================
# FINE-TUNING
# =========================================================
print("\n" + "=" * 70)
print("ENTRENAMIENTO / FINETUNING")
print("=" * 70)

train_result = trainer.train()

print("\n=== TRAIN RESULT ===")
print(train_result)

# =========================================================
# GUARDAR MODELO FINAL
# =========================================================
os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"\n[OK] Modelo guardado en: {OUTPUT_DIR}")

# =========================================================
# POST-FINETUNE
# =========================================================
print("\n" + "=" * 70)
print("POST-FINETUNE")
print("=" * 70)

post_valid_metrics, post_valid_df = evaluate_model_on_df(
    model=model,
    tokenizer=tokenizer,
    df=valid_df,
    split_name="VALID / POST-FINETUNE",
    batch_size=BATCH_SIZE_EVAL,
)

if test_df is not None:
    post_test_metrics, post_test_df = evaluate_model_on_df(
        model=model,
        tokenizer=tokenizer,
        df=test_df,
        split_name="TEST / POST-FINETUNE",
        batch_size=BATCH_SIZE_EVAL,
    )
else:
    post_test_metrics, post_test_df = None, None

print("\n=== EJEMPLOS / POST-FINETUNE ===")
for ex in examples:
    print(predict_sentiment(model, tokenizer, ex))

# =========================================================
# COMPARACIÓN FINAL
# =========================================================
print("\n" + "=" * 70)
print("COMPARACIÓN FINAL")
print("=" * 70)

comparison_rows = [
    {
        "split": "valid",
        "stage": "pre_finetune",
        "accuracy": baseline_valid_metrics["accuracy"],
        "f1_macro": baseline_valid_metrics["f1_macro"],
        "f1_weighted": baseline_valid_metrics["f1_weighted"],
    },
    {
        "split": "valid",
        "stage": "post_finetune",
        "accuracy": post_valid_metrics["accuracy"],
        "f1_macro": post_valid_metrics["f1_macro"],
        "f1_weighted": post_valid_metrics["f1_weighted"],
    },
]

if baseline_test_metrics is not None and post_test_metrics is not None:
    comparison_rows.extend([
        {
            "split": "test",
            "stage": "pre_finetune",
            "accuracy": baseline_test_metrics["accuracy"],
            "f1_macro": baseline_test_metrics["f1_macro"],
            "f1_weighted": baseline_test_metrics["f1_weighted"],
        },
        {
            "split": "test",
            "stage": "post_finetune",
            "accuracy": post_test_metrics["accuracy"],
            "f1_macro": post_test_metrics["f1_macro"],
            "f1_weighted": post_test_metrics["f1_weighted"],
        },
    ])

comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df)

print("\n=== MEJORA EN VALID ===")
print({
    "delta_accuracy": post_valid_metrics["accuracy"] - baseline_valid_metrics["accuracy"],
    "delta_f1_macro": post_valid_metrics["f1_macro"] - baseline_valid_metrics["f1_macro"],
    "delta_f1_weighted": post_valid_metrics["f1_weighted"] - baseline_valid_metrics["f1_weighted"],
})

# =========================================================
# OPCIONAL: MUESTRA ERRORES DEL POST-FINETUNE
# =========================================================
print("\n=== ALGUNOS ERRORES / VALID / POST-FINETUNE ===")
errors_df = post_valid_df[post_valid_df["gold"] != post_valid_df["pred"]].copy()
print(errors_df.head(20))

# =========================================================
# CÓMO CARGAR EL MODELO GUARDADO
# =========================================================
print("\n=== CARGA DEL MODELO GUARDADO ===")
print(f"MODEL_DIR = r'{OUTPUT_DIR}'")

[INFO] Descargando dataset: iberbench/iberbench_all / tass-tass-sentiment_analysis-2020-spanish-mexico

[INFO] Splits disponibles:
DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'language_variation', 'language'],
        num_rows: 989
    })
})

[INFO] Columna texto detectada: text
[INFO] Columna label detectada: label

[INFO] Labels únicos crudos en train:
['0', '1', '2']

TRAIN SHAPE: (890, 2)
VALID SHAPE: (99, 2)
TEST SHAPE : None

Distribución TRAIN:
label
negative    453
neutral     282
positive    155
Name: count, dtype: int64

Distribución VALID:
label
negative    51
neutral     31
positive    17
Name: count, dtype: int64


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Map:   0%|          | 0/890 [00:00<?, ? examples/s]

Map:   0%|          | 0/99 [00:00<?, ? examples/s]


BASELINE / PRE-FINETUNE

=== EVALUACIÓN MANUAL / VALID / PRE-FINETUNE ===
{'accuracy': 0.46464646464646464, 'f1_macro': 0.3922346259302781, 'f1_weighted': 0.4649713943192204}

=== CLASSIFICATION REPORT / VALID / PRE-FINETUNE ===
              precision    recall  f1-score   support

    negative     0.6341    0.5098    0.5652        51
     neutral     0.4186    0.5806    0.4865        31
    positive     0.1333    0.1176    0.1250        17

    accuracy                         0.4646        99
   macro avg     0.3954    0.4027    0.3922        99
weighted avg     0.4807    0.4646    0.4650        99


=== EJEMPLOS / PRE-FINETUNE ===
{'text': 'Qué chido quedó eso', 'raw_output': 'polarity', 'label': 'positive'}
{'text': 'Está de la fregada el servicio', 'raw_output': 'polaridad', 'label': 'positive'}
{'text': 'No está mal, cumple', 'raw_output': "i'm not a fan", 'label': 'negative'}

[INFO] Training args:
Seq2SeqTrainingArguments(
accelerator_config={'split_batches': False, 'dispatch

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


[OK] Modelo guardado en: /content/flan_t5_small_tass2020_mexico

POST-FINETUNE

=== EVALUACIÓN MANUAL / VALID / POST-FINETUNE ===
{'accuracy': 0.6868686868686869, 'f1_macro': 0.6082889856474761, 'f1_weighted': 0.67331271104856}

=== CLASSIFICATION REPORT / VALID / POST-FINETUNE ===
              precision    recall  f1-score   support

    negative     0.7455    0.8039    0.7736        51
     neutral     0.6286    0.7097    0.6667        31
    positive     0.5556    0.2941    0.3846        17

    accuracy                         0.6869        99
   macro avg     0.6432    0.6026    0.6083        99
weighted avg     0.6762    0.6869    0.6733        99


=== EJEMPLOS / POST-FINETUNE ===
{'text': 'Qué chido quedó eso', 'raw_output': 'positive', 'label': 'positive'}
{'text': 'Está de la fregada el servicio', 'raw_output': 'negative', 'label': 'negative'}
{'text': 'No está mal, cumple', 'raw_output': 'negative', 'label': 'negative'}

COMPARACIÓN FINAL
   split          stage  accuracy 

In [ ]:
import shutil
from google.colab import files

MODEL_DIR = "/content/flan_t5_small_tass2020_mexico"
ZIP_BASE = "/content/flan_t5_small_tass2020_mexico"

# Esto crea /content/flan_t5_small_tass2020_mexico.zip
# incluyendo la carpeta completa con su nombre
shutil.make_archive(
    base_name=ZIP_BASE,
    format="zip",
    root_dir="/content",
    base_dir="flan_t5_small_tass2020_mexico"
)

files.download(ZIP_BASE + ".zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_DIR = "/content/flan_t5_small_tass2020_mexico"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MAX_INPUT_LEN = 128
MAX_TARGET_LEN = 8

PREFIX = (
    "Clasifica la polaridad del siguiente comentario en español de México. "
    "Responde solo con una palabra exacta de esta lista: "
    "positive, neutral, negative. Tweet: "
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR).to(DEVICE)
model.eval()


def normalize_generated_label(text: str) -> str:
    z = str(text).strip().lower()
    z = z.replace(".", "").replace(",", "").replace(":", "").replace(";", "").strip()

    alias = {
        "positive": "positive",
        "positivo": "positive",
        "positiva": "positive",
        "pos": "positive",
        "p": "positive",

        "negative": "negative",
        "negativo": "negative",
        "negativa": "negative",
        "neg": "negative",
        "n": "negative",

        "neutral": "neutral",
        "neutro": "neutral",
        "neutra": "neutral",
        "neu": "neutral",
    }

    if z in alias:
        return alias[z]

    for k, v in alias.items():
        if k in z:
            return v

    return "neutral"


@torch.no_grad()
def predict_sentiment(text: str):
    enc = tokenizer(
        PREFIX + str(text),
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LEN,
    )
    enc = {k: v.to(DEVICE) for k, v in enc.items()}

    out = model.generate(
        **enc,
        max_new_tokens=MAX_TARGET_LEN,
        do_sample=False,
    )

    raw = tokenizer.decode(out[0], skip_special_tokens=True).strip()
    pred = normalize_generated_label(raw)

    return {
        "text": text,
        "raw_output": raw,
        "label": pred,
    }


# Ejemplos
print(predict_sentiment("Qué chido quedó eso"))
print(predict_sentiment("Está de la fregada el servicio"))
print(predict_sentiment("No está mal, cumple"))

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


{'text': 'Qué chido quedó eso', 'raw_output': 'positive', 'label': 'positive'}
{'text': 'Está de la fregada el servicio', 'raw_output': 'negative', 'label': 'negative'}
{'text': 'No está mal, cumple', 'raw_output': 'negative', 'label': 'negative'}
